# P300 speller (calibration mode)

- Target letter is known each time, so every falsh can be labelled as target or non-target

## Packages

- source .speller/bin/activate
- deactivate

- conda activate speller
- conda install -c conda-forge psychopy -y

- python 3.9 but also 3.10 works
- psychopy - conda install -c conda-forge "pyglet<2" -y
- pylsl

## LSL + Faces

- sequence is 12 flashses total 
- 1 row, 1 column falsh is target
- oher 10 are non-target

Markers
* **`experiment_start` / `experiment_end`**: mark the beginning and end of the whole speller session
* **`focus_start/<char>` / `focus_end/<char>`**: mark the period where the participant is shown which character to focus on
* **`target_start/<char>` / `target_end/<char>`**: mark the start and end of one target-character block
* **`sequence_start/<char>/<seq>` / `sequence_end/<char>/<seq>`**: mark the start and end of one full shuffled row+column flashing sequence
* **`flash_on/<kind>/<idx>/target_<0 or 1>/char_<char>/seq_<seq>/flash_<flash>`**: marks the exact onset of a flashed row or column, used to time-lock EEG epochs for P300 analysis
* **`flash_off/<kind>/<idx>/target_<0 or 1>/char_<char>/seq_<seq>/flash_<flash>`**: marks the end of that flash, mainly useful for timing checks and debugging
* **`target_1` vs `target_0`**: indicates whether the flashed row/column contains the attended target character, giving the label for CNN training
* **`kind` and `idx`**: specify which row or column was flashed, so predictions can later be combined into the final selected character



Full pipeline needed: 
1. Speller
- sends marker stream via LSL
2. EEG acquisition software 
- sends EEG stream via LSL
3. Recorder
- records EEG + markers into XDF
4. Processing
- reads the EEG marker timing, epochs the data, trains / runs CNN

In [ ]:
from psychopy import visual, core, event
from PIL import Image
import random
import os


# Parameters ---------------------
GRID_ROWS = 6
GRID_COLS = 6

# Flash timing (seconds)
FLASH_DUR = 0.100   # how long a row/col stays highlighted
ISI       = 0.075   # pause between flashes
N_SEQS    = 10      # how many full row+col sequences per target letter

# Window settings
WIN_SIZE = (1400, 900)
BG_COLOR = "black"
UNITS = "pix"

# Grid layout
X_SPACING = 180
Y_SPACING = 130

# Max displayed size of each image in pixels
CELL_MAX_W = 120
CELL_MAX_H = 140

# Colors
BOX_COLOR = "white"
HIGHLIGHT_COLOR = "yellow"
TEXT_COLOR = "white"

# Folder with your cell images
IMAGE_DIR = "Speller/Images"

# Word to spell during calibration
TARGET_WORD = "BRAIN"


# Helper functions ---------------------
def safe_quit(win):
    # Close PsychoPy cleanly
    win.close()
    core.quit()


def check_escape(win):
    # Allow quitting anytime with Escape
    keys = event.getKeys()
    if "escape" in keys:
        safe_quit(win)


def make_image_cell(win, img_path, pos, max_w=CELL_MAX_W, max_h=CELL_MAX_H):
    # Open image to get original pixel size
    pil_img = Image.open(img_path)
    orig_w, orig_h = pil_img.size

    # Preserve aspect ratio and fit inside max box
    scale = min(max_w / orig_w, max_h / orig_h)
    disp_w = int(orig_w * scale)
    disp_h = int(orig_h * scale)

    # Create image stimulus using the computed size
    image_stim = visual.ImageStim(
        win=win,
        image=img_path,
        pos=pos,
        size=(disp_w, disp_h),
        units=UNITS,
        interpolate=True
    )

    # Create border box with exactly the same size as image
    box_stim = visual.Rect(
        win=win,
        pos=pos,
        width=disp_w,
        height=disp_h,
        units=UNITS,
        lineColor=BOX_COLOR,
        fillColor=None,
        lineWidth=2
    )

    return image_stim, box_stim, disp_w, disp_h


def build_grid_positions(rows, cols, x_spacing, y_spacing):
    # Build centered grid positions in pixel units
    positions = []

    total_w = (cols - 1) * x_spacing
    total_h = (rows - 1) * y_spacing

    start_x = -total_w / 2
    start_y = total_h / 2

    for r in range(rows):
        for c in range(cols):
            x = start_x + c * x_spacing
            y = start_y - r * y_spacing
            positions.append((r, c, (x, y)))

    return positions


def draw_grid(cells, highlight_type=None, highlight_index=None):
    # Draw all cells and optionally highlight one row or column
    for cell in cells:
        row = cell["row"]
        col = cell["col"]

        is_highlighted = False
        if highlight_type == "row" and row == highlight_index:
            is_highlighted = True
        elif highlight_type == "col" and col == highlight_index:
            is_highlighted = True

        if is_highlighted:
            cell["box"].lineColor = HIGHLIGHT_COLOR
            cell["box"].lineWidth = 4
        else:
            cell["box"].lineColor = BOX_COLOR
            cell["box"].lineWidth = 2

        cell["box"].draw()
        cell["image"].draw()


def draw_message(win, text, y=0, height=28):
    # Draw text on screen
    msg = visual.TextStim(
        win=win,
        text=text,
        pos=(0, y),
        height=height,
        color=TEXT_COLOR,
        units=UNITS
    )
    msg.draw()


# Window ---------------------
win = visual.Window(
    size=WIN_SIZE,
    color=BG_COLOR,
    units=UNITS,
    fullscr=False
)


# Load image paths ---------------------
image_files = [f"cell_{i:02d}.jpg" for i in range(GRID_ROWS * GRID_COLS)]
image_paths = [os.path.join(IMAGE_DIR, f) for f in image_files]

# Check if all images exist
missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    print("Missing image files:")
    for m in missing:
        print(m)
    safe_quit(win)


# Build grid ---------------------
cells = []
positions = build_grid_positions(GRID_ROWS, GRID_COLS, X_SPACING, Y_SPACING)

for idx, (row, col, pos) in enumerate(positions):
    img_stim, box_stim, actual_w, actual_h = make_image_cell(
        win=win,
        img_path=image_paths[idx],
        pos=pos,
        max_w=CELL_MAX_W,
        max_h=CELL_MAX_H
    )

    cells.append({
        "index": idx,
        "row": row,
        "col": col,
        "pos": pos,
        "image": img_stim,
        "box": box_stim,
        "width": actual_w,
        "height": actual_h
    })


# Instructions screen ---------------------
while True:
    win.clearBuffer()

    draw_message(win, "Calibration Speller", y=380, height=36)
    draw_message(win, f"Spell the word: {TARGET_WORD}", y=330, height=28)
    draw_message(win, "Press SPACE to start, ESC to quit", y=-410, height=24)

    draw_grid(cells)
    win.flip()

    keys = event.waitKeys()
    if "space" in keys:
        break
    if "escape" in keys:
        safe_quit(win)


# Calibration loop ---------------------
for target_letter in TARGET_WORD:

    # Show current target letter
    instruction_timer = core.Clock()
    while instruction_timer.getTime() < 1.0:
        check_escape(win)
        win.clearBuffer()

        draw_message(win, f"Focus on target letter: {target_letter}", y=380, height=28)
        draw_message(win, "Stay focused on the flashes", y=-410, height=24)

        draw_grid(cells)
        win.flip()

    # Row/column flashing
    for seq in range(N_SEQS):
        flash_order = [("row", r) for r in range(GRID_ROWS)] + [("col", c) for c in range(GRID_COLS)]
        random.shuffle(flash_order)

        for flash_type, flash_idx in flash_order:
            check_escape(win)

            # Flash on
            win.clearBuffer()
            draw_message(win, f"Target: {target_letter}", y=380, height=28)
            draw_message(win, f"Sequence {seq + 1}/{N_SEQS}", y=-410, height=24)

            draw_grid(cells, highlight_type=flash_type, highlight_index=flash_idx)
            win.flip()
            core.wait(FLASH_DUR)

            # ISI
            check_escape(win)
            win.clearBuffer()
            draw_message(win, f"Target: {target_letter}", y=380, height=28)
            draw_message(win, f"Sequence {seq + 1}/{N_SEQS}", y=-410, height=24)

            draw_grid(cells)
            win.flip()
            core.wait(ISI)


# Wait at the end until you close it ---------------------
while True:
    win.clearBuffer()
    draw_grid(cells)
    draw_message(win, "Press ESC or Q to close", y=-410, height=24)
    win.flip()

    keys = event.getKeys()
    if "escape" in keys or "q" in keys:
        break

# Close PsychoPy cleanly ---------------------
win.close()
core.quit()

Fontconfig warning: ignoring UTF-8: not a valid region tag


Missing image files:
Speller/Images/cell_00.jpg
Speller/Images/cell_01.jpg
Speller/Images/cell_02.jpg
Speller/Images/cell_03.jpg
Speller/Images/cell_04.jpg
Speller/Images/cell_05.jpg
Speller/Images/cell_06.jpg
Speller/Images/cell_07.jpg
Speller/Images/cell_08.jpg
Speller/Images/cell_09.jpg
Speller/Images/cell_10.jpg
Speller/Images/cell_11.jpg
Speller/Images/cell_12.jpg
Speller/Images/cell_13.jpg
Speller/Images/cell_14.jpg
Speller/Images/cell_15.jpg
Speller/Images/cell_16.jpg
Speller/Images/cell_17.jpg
Speller/Images/cell_18.jpg
Speller/Images/cell_19.jpg
Speller/Images/cell_20.jpg
Speller/Images/cell_21.jpg
Speller/Images/cell_22.jpg
Speller/Images/cell_23.jpg
Speller/Images/cell_24.jpg
Speller/Images/cell_25.jpg
Speller/Images/cell_26.jpg
Speller/Images/cell_27.jpg
Speller/Images/cell_28.jpg
Speller/Images/cell_29.jpg
Speller/Images/cell_30.jpg
Speller/Images/cell_31.jpg
Speller/Images/cell_32.jpg
Speller/Images/cell_33.jpg
Speller/Images/cell_34.jpg
Speller/Images/cell_35.jpg
51.9608

SystemExit: 0

/opt/anaconda3/envs/speller/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 